In [1]:
############################################
#### 费舍尔信息计算 #############################
############################################

from datasets import load_dataset
import torch
import torch.nn.functional as F
import numpy as np


input_texts = [
    "The mitochondrion is the powerhouse of the cell, responsible for producing energy in the form of ATP.",  # 百科知识
    "To be or not to be, that is the question: Whether 'tis nobler in the mind to suffer The slings and arrows of outrageous fortune...",  # 诗歌（莎士比亚）
    "The capital of France is Paris, a city renowned for its art, culture, and history.",  # 常识
    "What is the meaning of life? Philosophers have debated this question for centuries, with no definitive answer.",  # 哲学问题
    "Once upon a time, in a faraway land, there was a brave knight who embarked on a quest to save the kingdom from an evil dragon.",  # 童话故事
    "你站在桥上看风景，看风景的人在楼上看你。明月装饰了你的窗子，你装饰了别人的梦。",  # 中文诗句（卞之琳《断章》）
    "Artificial intelligence is transforming the world by enabling machines to learn from data and make decisions autonomously.",  # 科技主题
    "Hey, how are you doing today? I hope everything is going well!",  # 日常对话
    "在那遥远的地方，有位美丽的姑娘。她的眼睛像星星一样明亮，她的笑容如阳光般温暖。",  # 歌词（王洛宾《在那遥远的地方》）
    "The theory of relativity was proposed by Albert Einstein, fundamentally changing our understanding of space and time.",  # 科学知识
    "Quantum mechanics describes the behavior of particles at the smallest scales, where classical physics breaks down.",  # 物理知识
    "Climate change poses a significant threat to global ecosystems and human societies. It is crucial to take action now to mitigate its effects.",  # 环境科学
    "In the midst of winter, I found there was, within me, an invincible summer.",  # 文学名言（阿尔贝·加缪）
    "机器学习是人工智能的一个分支，它使计算机能够在不进行明确编程的情况下从数据中学习并改进其性能。",  # 科技文章（中文）
    "The human brain contains approximately 86 billion neurons, each capable of forming thousands of connections.",  # 生物学知识
    "Dialogue systems, such as chatbots, are becoming increasingly sophisticated, thanks to advances in natural language processing.",  # 对话系统
    "A journey of a thousand miles begins with a single step. This ancient Chinese proverb reminds us of the importance of perseverance.",  # 谚语（中文）
    "The Great Wall of China stretches over 13,000 miles and is one of the most impressive architectural achievements in history.",  # 历史知识
    "The universe is vast and full of mysteries waiting to be discovered. From black holes to dark matter, there is still so much we do not know.",  # 天文学知识
    "Innovation distinguishes between a leader and a follower. Steve Jobs' words continue to inspire entrepreneurs around the world.",  # 商业名言
    "The process of photosynthesis allows plants to convert sunlight into energy, which is essential for their growth and survival.",  # 生物学知识
    "The Renaissance was a period of great cultural and artistic achievement in Europe, marked by the works of artists like Leonardo da Vinci and Michelangelo.",  # 历史知识
    "The Internet has revolutionized the way we communicate, making it possible to connect with people around the world in an instant.",  # 科技主题
    "The concept of democracy is based on the idea that power should be vested in the people, who have the right to choose their leaders and make decisions about their society.",  # 政治学知识
    "The Amazon rainforest is the largest tropical rainforest in the world, known for its incredible biodiversity and vital role in regulating the global climate.",  # 地理知识
    "The invention of the printing press by Johannes Gutenberg in the 15th century greatly facilitated the spread of knowledge and information.",  # 历史知识
]

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
from tqdm import tqdm

# 加载 Vicuna 模型和分词器
model_name = "/mnt/data2/zhuyao/models/vicuna_7b"  # 替换为你的模型路径
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')

# 确保模型在评估模式下
model.eval()

def compute_sensitivity_v2(model, input_texts, tokenizer, batch_size=4, loss_fn=torch.nn.CrossEntropyLoss()):
    """ 计算每个 Transformer Block 的敏感度，改进版：使用相对变化量 + 分段归一化 """
    model.eval()
    num_layers = len(model.model.layers)  # 获取 Transformer Block 层数
    sensitivity = np.zeros(num_layers)  # 初始化敏感度数组

    num_batches = (len(input_texts) + batch_size - 1) // batch_size

    for i in tqdm(range(0, len(input_texts), batch_size), desc="Computing Sensitivity"):
        batch_texts = input_texts[i:i + batch_size]

        # 处理 batch 输入
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_ids = inputs["input_ids"]

        # 正常前向传播
        model.zero_grad()
        output = model(input_ids).logits

        # 调整目标值形状
        target = input_ids[:, 1:].reshape(-1)
        logits = output[:, :-1, :].reshape(-1, output.shape[-1])
        loss = loss_fn(logits, target)
        loss.backward()

        # 遍历 Transformer Block
        batch_sensitivity = np.zeros(num_layers)

        for layer_idx, block in enumerate(model.model.layers):
            attn_norm, mlp_norm = 0.0, 0.0

            for name, param in block.named_parameters():
                if param.grad is not None:
                    grad_norm = torch.norm(param.grad, 'fro').item()
                    weight_norm = torch.norm(param.data, 'fro').item()
                    relative_sensitivity = grad_norm / (weight_norm + 1e-8)

                    # 归类
                    if "mlp" in name.lower():
                        # print('mlp')
                        mlp_norm += relative_sensitivity
                    elif "attn" in name.lower() or "self_attn" in name.lower():
                        attn_norm += relative_sensitivity
                        # print('attn')

            batch_sensitivity[layer_idx] = attn_norm + mlp_norm

        sensitivity += batch_sensitivity

    seg1, seg2, seg3, seg4 = np.split(sensitivity, [num_layers//4, 2*num_layers//4, 3*num_layers//4])
    # 对每段分别归一化
    seg1 /= seg1.sum() + 1e-8
    seg2 /= seg2.sum() + 1e-8
    seg3 /= seg3.sum() + 1e-8
    seg4 /= seg4.sum() + 1e-8
    # 合并
    sensitivity = np.concatenate([seg1, seg2, seg3, seg4])
    
    
    # return np.exp(-sensitivity)
    return 1/sensitivity


# ========== 计算层敏感度（逐 batch） ==========
batch_size = 8  # 可调整 batch size 以适应显存
sensitivity = compute_sensitivity_v2(model, input_texts, tokenizer, batch_size=batch_size)

print(sensitivity)

# 将多行字符串转换为单行并去除多余的空格
numbers = ' '.join(str(sensitivity)[1:-1].split())
# 将字符串分割成数字列表
numbers_list = numbers.split()
# 将数字列表转换为带有逗号分隔符的字符串，并保留六位小数
formatted_string = ', '.join(f"{float(num):.6f}" for num in numbers_list)

print(formatted_string)


/home/zhuyao/anaconda3/envs/SVD-LLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zhuyao/anaconda3/envs/SVD-LLM/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/zhuyao/anaconda3/envs/SVD-LLM/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/zhuyao/anaconda3/envs/SVD-LLM/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Plea

[ 2.98626845  3.76158279 11.25125558 15.3711208  13.05224219 18.78403581
 15.40485544 19.76836687  6.01237409  6.15983875  7.1414301   8.2134103
  9.54391004  9.92966623  9.71290982  9.89007972  6.063272    6.65403064
  6.25942767  8.12329021  9.16280051  9.63697703  9.98259894 11.25522557
  9.54700348 11.19413733  9.7344741   9.66874849  9.08034357  8.9294884
  5.00476643  5.62293987]
2.986268, 3.761583, 11.251256, 15.371121, 13.052242, 18.784036, 15.404855, 19.768367, 6.012374, 6.159839, 7.141430, 8.213410, 9.543910, 9.929666, 9.712910, 9.890080, 6.063272, 6.654031, 6.259428, 8.123290, 9.162801, 9.636977, 9.982599, 11.255226, 9.547003, 11.194137, 9.734474, 9.668748, 9.080344, 8.929488, 5.004766, 5.622940


In [2]:
############################################
#### 有效秩计算 #############################
############################################


from datasets import load_dataset
import torch
import torch.nn.functional as F
import numpy as np



input_texts = [
    "The mitochondrion is the powerhouse of the cell, responsible for producing energy in the form of ATP.",  # 百科知识
    "To be or not to be, that is the question: Whether 'tis nobler in the mind to suffer The slings and arrows of outrageous fortune...",  # 诗歌（莎士比亚）
    "The capital of France is Paris, a city renowned for its art, culture, and history.",  # 常识
    "What is the meaning of life? Philosophers have debated this question for centuries, with no definitive answer.",  # 哲学问题
    "Once upon a time, in a faraway land, there was a brave knight who embarked on a quest to save the kingdom from an evil dragon.",  # 童话故事
    "你站在桥上看风景，看风景的人在楼上看你。明月装饰了你的窗子，你装饰了别人的梦。",  # 中文诗句（卞之琳《断章》）
    "Artificial intelligence is transforming the world by enabling machines to learn from data and make decisions autonomously.",  # 科技主题
    "Hey, how are you doing today? I hope everything is going well!",  # 日常对话
    "在那遥远的地方，有位美丽的姑娘。她的眼睛像星星一样明亮，她的笑容如阳光般温暖。",  # 歌词（王洛宾《在那遥远的地方》）
    "The theory of relativity was proposed by Albert Einstein, fundamentally changing our understanding of space and time.",  # 科学知识
    "Quantum mechanics describes the behavior of particles at the smallest scales, where classical physics breaks down.",  # 物理知识
    "Climate change poses a significant threat to global ecosystems and human societies. It is crucial to take action now to mitigate its effects.",  # 环境科学
    "In the midst of winter, I found there was, within me, an invincible summer.",  # 文学名言（阿尔贝·加缪）
    "机器学习是人工智能的一个分支，它使计算机能够在不进行明确编程的情况下从数据中学习并改进其性能。",  # 科技文章（中文）
    "The human brain contains approximately 86 billion neurons, each capable of forming thousands of connections.",  # 生物学知识
    "Dialogue systems, such as chatbots, are becoming increasingly sophisticated, thanks to advances in natural language processing.",  # 对话系统
    "A journey of a thousand miles begins with a single step. This ancient Chinese proverb reminds us of the importance of perseverance.",  # 谚语（中文）
    "The Great Wall of China stretches over 13,000 miles and is one of the most impressive architectural achievements in history.",  # 历史知识
    "The universe is vast and full of mysteries waiting to be discovered. From black holes to dark matter, there is still so much we do not know.",  # 天文学知识
    "Innovation distinguishes between a leader and a follower. Steve Jobs' words continue to inspire entrepreneurs around the world.",  # 商业名言
    "The process of photosynthesis allows plants to convert sunlight into energy, which is essential for their growth and survival.",  # 生物学知识
    "The Renaissance was a period of great cultural and artistic achievement in Europe, marked by the works of artists like Leonardo da Vinci and Michelangelo.",  # 历史知识
    "The Internet has revolutionized the way we communicate, making it possible to connect with people around the world in an instant.",  # 科技主题
    "The concept of democracy is based on the idea that power should be vested in the people, who have the right to choose their leaders and make decisions about their society.",  # 政治学知识
    "The Amazon rainforest is the largest tropical rainforest in the world, known for its incredible biodiversity and vital role in regulating the global climate.",  # 地理知识
    "The invention of the printing press by Johannes Gutenberg in the 15th century greatly facilitated the spread of knowledge and information.",  # 历史知识
]

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm
import numpy as np

# 加载 Vicuna 模型和分词器
model_name = "/mnt/data2/zhuyao/models/vicuna_7b"  # 替换为你的模型路径
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')

# 确保模型在评估模式下
model.eval()


def compute_effective_rank(model, input_texts, tokenizer, batch_size=8, threshold=0.9):
    """
    计算每个 Transformer Block 输出的有效秩。
    参数：
        - model: LLM 模型
        - input_texts: 输入文本列表
        - tokenizer: 分词器
        - batch_size: 批次大小
        - threshold: 奇异值累积和阈值，例如 0.95 表示达到95%信息量
    返回：
        - ranks: 每层的有效秩分布
    """
    model.eval()
    num_layers = len(model.model.layers)
    ranks = np.zeros(num_layers)  # 存储每层的有效秩
    device = model.device

    for i in tqdm(range(0, len(input_texts), batch_size), desc="Computing Effective Ranks"):
        batch_texts = input_texts[i:i + batch_size]

        # 处理 batch 输入
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        
        hidden_states = outputs.hidden_states[1:]  # Transformer Block 的所有层输出
        # print(len(hidden_states))
        # 遍历每层计算有效秩
        for layer_idx, hidden in enumerate(hidden_states):
            # 将 [batch, seq_len, hidden_dim] -> [batch*seq_len, hidden_dim]
            layer_output = hidden.reshape(-1, hidden.size(-1)).detach().cpu().numpy()
            # SVD 分解
            U, S, Vt = np.linalg.svd(layer_output, full_matrices=False)
            # 计算累积奇异值占比
            cumulative_energy = np.cumsum(S) / np.sum(S)

            # 找到超过 threshold 的最小秩
            effective_rank = np.searchsorted(cumulative_energy, threshold) + 1
            ranks[layer_idx] += effective_rank
    # 平均每 batch 的秩
    ranks /= (len(input_texts) / batch_size)
    return ranks


# ========== 计算层敏感度 ==========
mi_sensitivity = compute_effective_rank(model, input_texts, tokenizer, threshold=0.95)

mi_sensitivity = (-mi_sensitivity + np.mean(mi_sensitivity))/np.std(mi_sensitivity)

print(len(mi_sensitivity))

# 将多行字符串转换为单行并去除多余的空格
numbers = ' '.join(str(mi_sensitivity)[1:-1].split())
# 将字符串分割成数字列表
numbers_list = numbers.split()
# 将数字列表转换为带有逗号分隔符的字符串，并保留六位小数
formatted_string = ', '.join(f"{float(num):.6f}" for num in numbers_list)
print(formatted_string)



Computing Effective Ranks: 100%|██████████| 4/4 [00:29<00:00,  7.41s/it]

32
0.349912, 3.181203, 2.344080, 1.720236, 1.256352, 0.952428, 0.755144, 0.595184, 0.440556, 0.280596, 0.173956, 0.099308, 0.008665, -0.071315, -0.167291, -0.273931, -0.348579, -0.428559, -0.497875, -0.545863, -0.583187, -0.636507, -0.657835, -0.679163, -0.705823, -0.737815, -0.775139, -0.807131, -0.855119, -0.892443, -1.089727, -1.404315


In [4]:
import numpy as np
from scipy.stats import pearsonr

# 目标剪枝比例
ratio1_8 = [0.59700192,0.76813825,0.9481952,0.9481952,0.9481952,0.9481952,0.85583155,0.9481952,0.77649692,0.71283235,0.65190775,0.78885188,0.86020153,0.83651402,0.85595998,0.9481952,0.87400903,0.92464963,0.90996209,0.8610263,0.85455071,0.76273061,0.72712014,0.73447863,0.69648933,0.67512856,0.72309583,0.6768789,0.78920115,0.6409697,0.72383763,0.63296439]
ratio1_7 = [0.50175422,0.80179475,0.85237401,0.95706593,0.86923101,0.90453245,0.76802439,0.83320083,0.66875449,0.69144545,0.50175422,0.71233495, 0.85692712,0.75530431,0.84008004,0.86784305,0.8890871 ,0.65136504,0.75840017,0.74614005,0.67271762,0.58973022,0.67143917,0.58411168,0.50175422,0.50175422,0.61646076,0.60165664,0.50175422,0.50175422,0.72769921,0.50175422]
ratio1_6 = [0.45852712,0.74137466,0.8140282,0.95198754,0.79192939, 0.87164759,0.73668272,0.75307537,0.64138094,0.45163508,0.57632983,0.55420449,0.78123887,0.68206394,0.75298724,0.80408131,0.83363316,0.52508567,0.61511184,0.65716213,0.52077019,0.4518841,0.5080388,0.47752183,0.39450214,0.43973719,0.45109401,0.37521535,0.44683538,0.24721279,0.47388439,0.41913674]

# 生成剪枝比例的函数
def generate_prune_ratio(S, U, beta):
    # 强制将数据转换为 NumPy 数组
    S = np.asarray(S, dtype=np.float64)
    U = np.asarray(U, dtype=np.float64)

    # 归一化
    S_norm = (S- S.min()+1e-5) / (S.max() - S.min())
    U_norm = (U- U.min()+1e-5) / (U.max() - U.min())

    # 加权相乘
    W = (S_norm ** beta) * (U_norm ** (1 - beta))
    # W = (S_norm ** beta) - (U_norm * (beta))
    return W

# 调整剪枝比例的函数
def scale_array_to_ratio(A, ratio):
    # 找到数组的最小值和最大值
    A_min = np.min(A)
    A_max = np.max(A)
    # 计算数组的范围
    R = A_max - A_min
    # 将数组归一化到 [0, 1] 区间
    A_normalized = (A - A_min) / R
    # 将归一化后的数组放缩到 [-ratio, +ratio] 区间
    A_scaled = -ratio + 2 * ratio * A_normalized
    return A_scaled

# 归一化数组使其和为指定值的函数
def normalize_array_to_sum(A, target_sum, max_iter=25, tol=1e-5):
    A = np.array(A, dtype=float)  # 确保数组是浮点类型
    current_sum = np.sum(A)
    if current_sum == 0:
        return np.zeros_like(A)  # 如果总和为0，直接返回全0数组
    # 初始化归一化数组
    A_normalized = A.copy()
    for _ in range(max_iter):
        # 计算当前总和
        current_sum = np.sum(A_normalized)
        # 如果当前总和已经接近目标总和，停止迭代
        if abs(current_sum - target_sum) < tol:
            break
        # 计算调整比例
        scale_factor = target_sum / current_sum
        # 调整数组值
        A_normalized *= scale_factor
        # 确保每个元素在 [0, 1] 范围内
        A_normalized = np.clip(A_normalized, 0.3, 0.98)
    # 如果经过多次迭代仍未满足条件，进行最后的调整
    if abs(np.sum(A_normalized) - target_sum) > tol:
        remaining_sum = target_sum - np.sum(A_normalized)
        # 按比例分配剩余的差值
        A_normalized += remaining_sum * (A_normalized / np.sum(A_normalized))
        # 再次确保每个元素在 [0, 1] 范围内
        A_normalized = np.clip(A_normalized, 0, 1)
    return A_normalized

# 网格搜索最佳 beta 的函数
def grid_search_beta(sensitivity, mi_sensitivity, ratio1_8, ratio1_7, ratio1_6, beta_range, target_sum, ratio):
    best_beta = 0
    best_score = -1  # 初始化为最小可能值
    results = []

    for beta in beta_range:
        # 生成剪枝比例
        W = generate_prune_ratio(sensitivity, mi_sensitivity, beta)
        # print("W:",W,beta)
        # 调整剪枝比例
        A_scaled = scale_array_to_ratio(W, ratio) + target_sum
        
        # print("A_scaled:",A_scaled)
        # 归一化剪枝比例
        S = normalize_array_to_sum(A_scaled, target_sum * len(W))
        
        # 计算与每个目标剪枝比例的皮尔逊相关系数
        corr_8, _ = pearsonr(ratio1_8, S)
        corr_7, _ = pearsonr(ratio1_7, S)
        corr_6, _ = pearsonr(ratio1_6, S)
        
        # 计算综合评分（取平均值）
        score = (corr_8 + corr_7 + corr_6) / 3
        
        # 记录结果
        results.append((beta, score, corr_8, corr_7, corr_6))
        
        # 更新最佳 beta
        if score > best_score:
            best_score = score
            best_beta = beta

    return best_beta, best_score, results

# 参数设置
target_sum = 0.8  # 目标总和;保留参数量
ratio = 0.2  # 调整比例
beta = 0.25

W = generate_prune_ratio(sensitivity, mi_sensitivity, beta)
A_scaled = scale_array_to_ratio(W, ratio) + target_sum
# 归一化剪枝比例
S = normalize_array_to_sum(A_scaled, target_sum * len(W))

# 输出结果
print("Best score (average correlation):", S)

# 格式化输出剪枝比例
formatted_string = ', '.join(f"{num:.6f}" for num in S)
print("Final pruning ratio:", formatted_string)

Best score (average correlation): [0.63938527 0.90251887 0.98       0.98       0.97351883 0.98
 0.93972514 0.94518536 0.82393058 0.81342671 0.81679368 0.82073373
 0.82264698 0.81710651 0.80560391 0.79528151 0.75863352 0.75671812
 0.74666024 0.7552444  0.75683023 0.75289938 0.75187626 0.75431328
 0.74419777 0.74657084 0.73643693 0.73216192 0.72387758 0.71857963
 0.67763589 0.63150524]
Final pruning ratio: 0.639385, 0.902519, 0.980000, 0.980000, 0.973519, 0.980000, 0.939725, 0.945185, 0.823931, 0.813427, 0.816794, 0.820734, 0.822647, 0.817107, 0.805604, 0.795282, 0.758634, 0.756718, 0.746660, 0.755244, 0.756830, 0.752899, 0.751876, 0.754313, 0.744198, 0.746571, 0.736437, 0.732162, 0.723878, 0.718580, 0.677636, 0.631505


0.6999998290988804 0.98 0.15017283190859765


In [5]:
# 相关性验证  Vicuna模型

from scipy.stats import pearsonr, spearmanr, kendalltau

## 贝叶斯优化得到的
ratio1_8 = [0.59700192,0.76813825,0.9481952,0.9481952,0.9481952,0.9481952,0.85583155,0.9481952,0.77649692,0.71283235,0.65190775,0.78885188,0.86020153,0.83651402,0.85595998,0.9481952,0.87400903,0.92464963,0.90996209,0.8610263,0.85455071,0.76273061,0.72712014,0.73447863,0.69648933,0.67512856,0.72309583,0.6768789,0.78920115,0.6409697,0.72383763,0.63296439]
ratio1_7 = [0.50175422,0.80179475,0.85237401,0.95706593,0.86923101,0.90453245,0.76802439,0.83320083,0.66875449,0.69144545,0.50175422,0.71233495, 0.85692712,0.75530431,0.84008004,0.86784305,0.8890871 ,0.65136504,0.75840017,0.74614005,0.67271762,0.58973022,0.67143917,0.58411168,0.50175422,0.50175422,0.61646076,0.60165664,0.50175422,0.50175422,0.72769921,0.50175422]
ratio1_6 = [0.45852712,0.74137466,0.8140282,0.95198754,0.79192939, 0.87164759,0.73668272,0.75307537,0.64138094,0.45163508,0.57632983,0.55420449,0.78123887,0.68206394,0.75298724,0.80408131,0.83363316,0.52508567,0.61511184,0.65716213,0.52077019,0.4518841,0.5080388,0.47752183,0.39450214,0.43973719,0.45109401,0.37521535,0.44683538,0.24721279,0.47388439,0.41913674]
ratio1_5 = [0.37078173, 0.5503113, 0.622523, 0.680297, 0.621275, 0.64866077, 0.562033, 0.603445, 0.496817, 0.441884, 0.41190281, 0.48937889, 0.5948494, 0.54140054, 0.58310173, 0.62383799, 0.618268879, 0.500261986, 0.54368431, 0.53912582889, 0.48762822, 0.4296059, 0.45395193, 0.42764574787, 0.3792251645, 0.3849095, 0.42634538, 0.3937502, 0.4137597, 0.3309373, 0.4584336265, 0.3699655597]


## 计算得到的
ratio2_6 = S
print(np.min(ratio1_5),np.max(ratio1_5),np.mean(ratio1_5),np.std(ratio1_5))
print(np.min(ratio1_6),np.max(ratio1_6),np.mean(ratio1_6),np.std(ratio1_6))
print(np.min(ratio2_6),np.max(ratio2_6),np.mean(ratio2_6),np.std(ratio2_6))

# pearson_corr, p_value = pearsonr(ratio1_8, ratio1_6)
# print(f"Pearson correlation coefficient: {pearson_corr}, p-value: {p_value}")

pearson_corr, p_value = pearsonr(ratio1_8, ratio2_6)
print(f"Pearson correlation coefficient: {pearson_corr}, p-value: {p_value}")

pearson_corr, p_value = pearsonr(ratio1_7, ratio2_6)
print(f"Pearson correlation coefficient: {pearson_corr}, p-value: {p_value}")

pearson_corr, p_value = pearsonr(ratio1_6, ratio2_6)
print(f"Pearson correlation coefficient: {pearson_corr}, p-value: {p_value}")


0.3309373 0.680297 0.499999949764375 0.09542850504504528
0.24721279 0.95198754 0.6 0.171935855401243
0.6315052350705014 0.98 0.7999999466689312 0.0958283146934416
Pearson correlation coefficient: 0.6868886020875139, p-value: 1.414294873079692e-05
Pearson correlation coefficient: 0.7157576844486174, p-value: 4.121359935183678e-06
Pearson correlation coefficient: 0.7725862050218025, p-value: 2.210160443844774e-07
